# [과제4] 스트리밍 알고리즘 비교 실험
## 스트리밍 알고리즘 2종 구현 및 정확도·메모리 트레이드오프 분석

- 과목명: 빅데이터분석 01분반
- 학번: 2342803
- 이름: 문서영

### 실험 환경

본 실험은 Google Colab 환경에서 Python을 사용하여 진행하였다.

- 실행 환경: Google Colab
- 프로그래밍 언어: Python
- Python 버전: 3.13.18
- 데이터셋: MovieLens 1M (`ratings.dat`)
- 데이터 레코드 수: 1,000,209개
- 데이터 처리 방식: 파일을 한 줄씩 읽는 스트리밍 방식
- 구현 알고리즘: Bloom Filter, Count-Min Sketch

---

##### Python 버전 및 실험환경

In [ ]:
import sys

print("Python 버전:", sys.version)
print("실험 환경: Google Colab")

Python 버전: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
실험 환경: Google Colab



## 1. 과제 목표

본 과제에서는 대용량 데이터 스트림 환경에서 전체 데이터를 메모리에 저장하지 않고 근사적인 계산을 수행하는 스트리밍 알고리즘을 구현한다.

Bloom Filter와 Count-Min Sketch를 직접 구현하고, 정확한 결과(Ground Truth)와 비교하여 정확도, 메모리 사용량, 처리시간 측면에서 성능을 분석한다.

또한 알고리즘의 파라미터를 변경하며 정확도와 메모리 사이의 Trade-off를 확인하고, 실제 서비스 로그 분석 환경에서의 활용 가능성을 살펴본다.

## 2. 데이터셋 준비 및 업로드

본 실험에서는 MovieLens 1M 데이터셋을 사용한다.
ratings.dat 파일을 업로드한 후, 각 평점 기록을 한 줄씩 읽어 스트리밍 알고리즘의 입력 데이터로 활용한다.

### 2-1. 데이터 파일 업로드

In [ ]:

from google.colab import files

# ratings.dat 파일 업로드
uploaded = files.upload()

# 업로드된 파일 이름 저장
DATA_PATH = next(iter(uploaded.keys()))

print("파일 업로드 완료!")
print("파일 이름:", DATA_PATH)


Saving ratings.dat to ratings (1).dat
파일 업로드 완료!
파일 이름: ratings (1).dat


### 2-2. 데이터 구조 및 레코드 수 확인

MovieLens 1M의 ratings.dat 파일은 사용자 ID, 영화 ID, 평점, 타임스탬프 순서로 구성되어 있다.

각 데이터는 `::` 구분자로 분리되며, 파일을 한 줄씩 읽어 전체 레코드 수와 데이터 형식을 확인한다.

In [ ]:
# 데이터 파일의 전체 레코드 수와 앞부분 확인
record_count = 0
sample_records = []

with open(DATA_PATH, "r", encoding="latin-1") as file:
    for line in file:
        record_count += 1

        if len(sample_records) < 5:
            sample_records.append(line.strip())

print("전체 레코드 수:", f"{record_count:,}개")
print("\n데이터 샘플 (앞의 5개)")
for record in sample_records:
    print(record)

전체 레코드 수: 1,000,209개

데이터 샘플 (앞의 5개)
1::1193::5::978300760
1::661::3::978302109
1::914::3::978301968
1::3408::4::978300275
1::2355::5::978824291


## 3. 스트리밍 알고리즘 입력 데이터 설정


본 실험에서는 MovieLens 데이터의 영화 ID(MovieID)를 스트림 데이터로 사용한다.

각 평점 기록에서 영화 ID를 추출하여 Bloom Filter와 Count-Min Sketch에 순차적으로 입력한다.

Bloom Filter는 영화 ID의 중복 여부를 확인하고, Count-Min Sketch는 영화 ID의 등장 빈도를 추정한다.

In [ ]:
# 영화 ID 추출 및 기본 통계 확인
movie_id_set = set()
record_count = 0

with open(DATA_PATH, "r", encoding="latin-1") as file:
    for line in file:
        parts = line.strip().split("::")

        if len(parts) == 4:
            movie_id = int(parts[1])
            movie_id_set.add(movie_id)
            record_count += 1

print("전체 데이터 레코드 수:", f"{record_count:,}개")
print("고유 영화 ID 수:", f"{len(movie_id_set):,}개")
print("영화 ID 샘플:", sorted(movie_id_set)[:10])

전체 데이터 레코드 수: 1,000,209개
고유 영화 ID 수: 3,706개
영화 ID 샘플: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 4. Bloom Filter 구현 및 실험

### 4-1. Bloom Filter 작동 원리

Bloom Filter는 데이터의 존재 여부를 빠르게 확인하는 확률적 자료구조이다.

1. 입력된 영화 ID에 여러 개의 해시 함수를 적용하여 비트 배열의 위치를 계산한다.
2. 계산된 위치의 비트를 1로 설정하여 데이터를 삽입한다.
3. 데이터의 존재 여부를 확인할 때 동일한 해시 함수를 적용한다.
4. 확인한 위치의 비트가 모두 1이면 해당 데이터가 존재할 가능성이 있다고 판단하고, 하나라도 0이면 존재하지 않는다고 판단한다.

Bloom Filter는 적은 메모리로 빠르게 데이터를 확인할 수 있지만, 실제로 존재하지 않는 데이터도 존재한다고 판단하는 False Positive(거짓 양성)가 발생할 수 있다는 특징이 있다.

### 4-2. Bloom Filter 클래스 구현

예상 입력 데이터 수와 목표 False Positive 확률을 이용하여 비트 배열의 크기와 해시 함수 개수를 계산한다.

해시 함수는 hashlib의 blake2b를 활용하고, 비트 배열은 bytearray로 구현한다.

In [ ]:
import math
import hashlib
import sys
import time

class BloomFilter:
    def __init__(self, expected_items, false_positive_rate=0.01):
        self.n = expected_items
        self.p = false_positive_rate

        # 비트 배열 크기 계산
        self.m = math.ceil(
            -self.n * math.log(self.p) / (math.log(2) ** 2)
        )

        # 최적 해시 함수 개수 계산
        self.k = round((self.m / self.n) * math.log(2))

        # 비트 배열 생성
        self.bit_array = bytearray((self.m + 7) // 8)

    # 여러 해시 위치 계산
    def get_positions(self, item):
        data = str(item).encode("utf-8")
        digest = hashlib.blake2b(data, digest_size=16).digest()

        h1 = int.from_bytes(digest[:8], "big")
        h2 = int.from_bytes(digest[8:], "big") | 1

        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    # 데이터 삽입
    def add(self, item):
        for position in self.get_positions(item):
            byte_index = position // 8
            bit_index = position % 8

            self.bit_array[byte_index] |= (1 << bit_index)

    # 데이터 존재 여부 확인
    def contains(self, item):
        for position in self.get_positions(item):
            byte_index = position // 8
            bit_index = position % 8

            if not (self.bit_array[byte_index] & (1 << bit_index)):
                return False

        return True


print("Bloom Filter 클래스 구현 완료!")

Bloom Filter 클래스 구현 완료!


### 4-3. Bloom Filter 실행 및 정확도 측정

전체 ratings.dat 파일을 한 줄씩 읽으면서 영화 ID를 Bloom Filter에 삽입한다.

실험이 끝난 후 실제로 등장한 영화 ID와 등장하지 않은 영화 ID를 각각 조회한다.

등장하지 않은 영화 ID를 존재한다고 잘못 판단한 비율을 False Positive Rate(FPR)로 계산한다.

In [ ]:
# Bloom Filter 생성
bloom = BloomFilter(
    expected_items=len(movie_id_set),
    false_positive_rate=0.01
)

# 스트리밍 방식으로 데이터 삽입 및 처리시간 측정
start_time = time.perf_counter()

processed_count = 0

with open(DATA_PATH, "r", encoding="latin-1") as file:
    for line in file:
        parts = line.strip().split("::")

        if len(parts) == 4:
            movie_id = int(parts[1])
            bloom.add(movie_id)
            processed_count += 1

processing_time = time.perf_counter() - start_time

# 실제 존재하는 영화 ID의 조회 결과
true_positive = sum(
    1 for movie_id in movie_id_set
    if bloom.contains(movie_id)
)

# 실제 존재하지 않는 영화 ID를 대상으로 False Positive 측정
non_member_ids = [
    movie_id for movie_id in range(1, 5001)
    if movie_id not in movie_id_set
]

false_positive = sum(
    1 for movie_id in non_member_ids
    if bloom.contains(movie_id)
)

fpr = false_positive / len(non_member_ids)

# 비트 배열 메모리 사용량
memory_bytes = sys.getsizeof(bloom.bit_array)

print("===== Bloom Filter 실험 결과 =====")
print("처리 레코드 수:", f"{processed_count:,}개")
print("비트 배열 크기:", f"{bloom.m:,} bits")
print("해시 함수 개수:", bloom.k)
print("실제 존재하는 영화 ID:", f"{len(movie_id_set):,}개")
print("True Positive:", true_positive)
print("False Positive:", false_positive)
print("False Positive Rate:", f"{fpr:.4%}")
print("비트 배열 메모리:", f"{memory_bytes:,} bytes")
print("처리 시간:", f"{processing_time:.4f}초")
print("처리 속도:", f"{processed_count / processing_time:,.0f} records/sec")

===== Bloom Filter 실험 결과 =====
처리 레코드 수: 1,000,209개
비트 배열 크기: 35,523 bits
해시 함수 개수: 7
실제 존재하는 영화 ID: 3,706개
True Positive: 3706
False Positive: 15
False Positive Rate: 1.1592%
비트 배열 메모리: 4,498 bytes
처리 시간: 5.5140초
처리 속도: 181,396 records/sec


### 4-4. Bloom Filter 실험 결과 분석

Bloom Filter를 이용하여 MovieLens 1M 데이터셋의 1,000,209개 레코드를 처리한 결과, 실제 고유 영화 ID 3,706개를 모두 정상적으로 탐지하였다.

실험에서 True Positive는 3,706개, False Positive는 15개로 나타났으며, False Positive Rate는 1.1592%로 측정되었다. 이는 목표로 설정한 1%보다 약간 높은 수치로, 이론적 FPR과 실제 측정값 사이에 차이가 발생했음을 보여준다.

메모리 사용량은 4,498 bytes로 약 4.4KB에 불과했으며, 전체 데이터 처리 시간은 5.5140초였다. 처리 속도는 초당 약 181,396개의 레코드로 나타났다.

이러한 결과를 통해 Bloom Filter는 적은 메모리로 대용량 데이터의 존재 여부를 빠르게 확인할 수 있다는 장점을 확인하였다. 다만 False Positive가 발생할 수 있으므로 정확한 데이터 존재 여부를 보장해야 하는 상황에는 주의가 필요하다.

또한 False Positive Rate는 실제로 존재하지 않는 영화 ID 중 1부터 5,000까지의 범위에서 조회한 ID를 대상으로 측정한 결과이다.

### 4-5. Bloom Filter 이론적 FPR과 실제 FPR 비교

Bloom Filter는 비트 배열의 크기와 해시 함수 개수에 따라 이론적인 False Positive Rate(FPR)가 결정된다.

실제 실험에서 측정한 FPR과 이론적으로 계산한 FPR을 비교하여 Bloom Filter의 오탐률이 예상한 수준과 얼마나 차이가 있는지 확인한다.

##### - 이론적 FPR 계산 코드



In [ ]:

# Bloom Filter 이론적 FPR 계산

theoretical_fpr = (
    1 - math.exp(
        -bloom.k * len(movie_id_set) / bloom.m
    )
) ** bloom.k

print("===== Bloom Filter FPR 비교 =====")
print(f"이론적 FPR: {theoretical_fpr:.4%}")
print(f"실제 측정 FPR: {fpr:.4%}")
print(f"실제 테스트 대상 수: {len(non_member_ids):,}개")
print(f"False Positive 수: {false_positive:,}개")


===== Bloom Filter FPR 비교 =====
이론적 FPR: 1.0038%
실제 측정 FPR: 1.1592%
실제 테스트 대상 수: 1,294개
False Positive 수: 15개


##### - Bloom Filter 이론적 FPR과 실제 FPR 비교 분석


Bloom Filter의 이론적 False Positive Rate는 1.0038%로 계산되었으며, 실제 실험에서 측정된 FPR은 1.1592%로 나타났다. 두 값의 차이는 약 0.1554%p로, 이론적으로 예상한 오탐률과 실제 측정한 오탐률이 비교적 유사한 수준임을 확인하였다.

실제 실험에서는 존재하지 않는 영화 ID 1,294개를 조회하였으며, 이 중 15개를 실제로 존재하는 데이터로 잘못 판단하였다. 이는 약 1.1592%의 오탐률에 해당한다.

이론적 FPR과 실제 FPR 사이에 차이가 발생한 이유는 이론적 계산이 이상적인 해시 함수의 분포를 가정한 근사값인 반면, 실제 실험은 제한된 개수의 조회 데이터와 해시 결과를 바탕으로 측정했기 때문이다.

이를 통해 Bloom Filter의 이론적 오탐률과 실제 데이터에서 측정한 오탐률을 비교하고, 목표 오탐률을 설정하더라도 실제 결과에는 일정한 차이가 발생할 수 있음을 확인하였다.

### 4-6. Bloom Filter 파라미터 변경에 따른 성능 비교

Bloom Filter의 목표 False Positive Rate를 5%, 1%, 0.1%로 변경하여 파라미터에 따른 성능 차이를 비교한다.

각 설정에서 비트 배열 크기, 해시 함수 개수, 실제 False Positive Rate, 메모리 사용량 및 처리 시간을 측정한다.

이를 통해 목표 오탐률을 낮추는 것이 메모리 사용량과 처리 성능에 어떤 영향을 미치는지 분석한다.

##### -  Bloom Filter 파라미터별 성능 비교 실험

In [ ]:

# Bloom Filter 파라미터별 성능 비교 실험

target_fpr_list = [0.05, 0.01, 0.001]

bloom_results = []

for target_fpr in target_fpr_list:

    # 목표 FPR에 따른 Bloom Filter 생성
    test_bloom = BloomFilter(
        expected_items=len(movie_id_set),
        false_positive_rate=target_fpr
    )

    # 스트리밍 방식으로 데이터 삽입 및 처리 시간 측정
    start_time = time.perf_counter()

    processed_count = 0

    with open(DATA_PATH, "r", encoding="latin-1") as file:
        for line in file:
            parts = line.strip().split("::")

            if len(parts) == 4:
                movie_id = int(parts[1])
                test_bloom.add(movie_id)
                processed_count += 1

    processing_time = time.perf_counter() - start_time

    # 실제 False Positive 측정
    false_positive_count = sum(
        1 for movie_id in non_member_ids
        if test_bloom.contains(movie_id)
    )

    actual_fpr = false_positive_count / len(non_member_ids)

    # 이론적 FPR 계산
    theoretical_fpr = (
        1 - math.exp(
            -test_bloom.k * len(movie_id_set) / test_bloom.m
        )
    ) ** test_bloom.k

    # 메모리 사용량
    memory_bytes = sys.getsizeof(test_bloom.bit_array)

    # 결과 저장
    bloom_results.append({
        "목표 FPR": target_fpr,
        "비트 배열 크기(bits)": test_bloom.m,
        "해시 함수 개수": test_bloom.k,
        "이론적 FPR": theoretical_fpr,
        "실제 FPR": actual_fpr,
        "False Positive": false_positive_count,
        "메모리(bytes)": memory_bytes,
        "처리 시간(초)": processing_time,
        "처리 속도(records/sec)": processed_count / processing_time
    })

# 결과 출력
print("===== Bloom Filter 파라미터 비교 결과 =====")

for result in bloom_results:
    print(f"\n목표 FPR: {result['목표 FPR']:.1%}")
    print(f"비트 배열 크기(bits): {result['비트 배열 크기(bits)']:,} bits")
    print(f"해시 함수 개수: {result['해시 함수 개수']}")
    print(f"이론적 FPR: {result['이론적 FPR']:.4%}")
    print(f"실제 FPR: {result['실제 FPR']:.4%}")
    print(f"False Positive: {result['False Positive']}개")
    print(f"메모리 사용량: {result['메모리(bytes)']:,} bytes")
    print(f"처리 시간: {result['처리 시간(초)']:.4f}초")
    print(f"처리 속도: {result['처리 속도(records/sec)']:,.0f} records/sec")


===== Bloom Filter 파라미터 비교 결과 =====

목표 FPR: 5.0%
비트 배열 크기(bits): 23,108 bits
해시 함수 개수: 4
이론적 FPR: 5.0268%
실제 FPR: 4.8686%
False Positive: 63개
메모리 사용량: 2,946 bytes
처리 시간: 5.8534초
처리 속도: 170,878 records/sec

목표 FPR: 1.0%
비트 배열 크기(bits): 35,523 bits
해시 함수 개수: 7
이론적 FPR: 1.0038%
실제 FPR: 1.1592%
False Positive: 15개
메모리 사용량: 4,498 bytes
처리 시간: 5.4798초
처리 속도: 182,527 records/sec

목표 FPR: 0.1%
비트 배열 크기(bits): 53,284 bits
해시 함수 개수: 10
이론적 FPR: 0.1000%
실제 FPR: 0.3091%
False Positive: 4개
메모리 사용량: 6,718 bytes
처리 시간: 8.0828초
처리 속도: 123,745 records/sec


##### - Bloom Filter 파라미터별 실험 결과

목표 False Positive Rate를 5%, 1%, 0.1%로 변경하여 실험한 결과는 다음과 같다.

In [ ]:
import pandas as pd

bloom_df = pd.DataFrame(bloom_results)

display(
    bloom_df.style
    .hide(axis="index")
    .format({
        "목표 FPR": "{:.1%}",
        "비트 배열 크기(bits)": "{:,}",
        "해시 함수 개수": "{:,}",
        "이론적 FPR": "{:.4%}",
        "실제 FPR": "{:.4%}",
        "False Positive": "{:,}",
        "메모리(bytes)": "{:,}",
        "처리 시간(초)": "{:.4f}",
        "처리 속도(records/sec)": "{:,.0f}"
    })
)

목표 FPR,비트 배열 크기(bits),해시 함수 개수,이론적 FPR,실제 FPR,False Positive,메모리(bytes),처리 시간(초),처리 속도(records/sec)
5.0%,"23,108",4,5.0268%,4.8686%,63,"2,946",5.8534,"170,878"
1.0%,"35,523",7,1.0038%,1.1592%,15,"4,498",5.4798,"182,527"
0.1%,"53,284",10,0.1000%,0.3091%,4,"6,718",8.0828,"123,745"


### 4-7. Bloom Filter 파라미터 변경 실험 결과 분석

실험 결과 목표 False Positive Rate(FPR)를 5%에서 0.1%로 낮출수록 실제 오탐률도 감소하는 경향을 확인하였다.

목표 FPR이 5%일 때 실제 FPR은 4.8686%였으며, 비트 배열 크기는 23,108 bits, 해시 함수 개수는 4개였다. 목표 FPR을 1%로 설정했을 때 실제 FPR은 1.1592%로 감소하였고, 비트 배열 크기는 35,523 bits, 해시 함수 개수는 7개로 증가하였다.

목표 FPR을 0.1%로 설정한 경우 실제 FPR은 0.3091%로 가장 낮게 나타났다. 이때 비트 배열 크기는 53,284 bits, 해시 함수 개수는 10개였으며, 메모리 사용량은 6,718 bytes로 가장 크게 나타났다.

메모리 사용량은 목표 FPR이 5%일 때 2,946 bytes에서 0.1%일 때 6,718 bytes로 증가하였다. 이는 오탐률을 낮추기 위해 더 큰 비트 배열과 더 많은 해시 함수를 사용했기 때문이다.

처리 시간은 각각 5.8534초, 5.4798초, 8.0828초로 측정되었다. 목표 FPR이 낮아질수록 처리 시간이 일정하게 증가하는 패턴은 나타나지 않았다. 다만 해시 함수 개수가 증가하면 레코드 처리에 필요한 연산이 많아질 수 있으며, 실제 처리 시간은 실행 환경과 시스템 상태의 영향을 받을 수 있다.

다만 이론적 FPR과 실제 FPR은 완전히 일치하지 않았다. 특히 목표 FPR이 0.1%일 때 실제 FPR은 0.3091%로 나타났다. 이는 실제 FPR이 존재하지 않는 영화 ID 1,294개를 대상으로 측정한 값이기 때문에 테스트 데이터의 범위와 개수에 따라 차이가 발생할 수 있기 때문이다.

이번 실험을 통해 Bloom Filter는 오탐률을 낮추도록 설정할수록 일반적으로 메모리 사용량과 처리 비용이 증가한다는 Trade-off를 확인하였다.

## 5. Count-Min Sketch 구현 및 실험

Count-Min Sketch는 데이터 스트림에서 각 데이터의 등장 빈도를 근사적으로 계산하는 확률적 자료구조이다.

여러 개의 해시 함수를 사용하여 2차원 카운터 배열의 위치를 계산하고, 입력 데이터가 들어올 때마다 해당 위치의 카운터를 증가시킨다.

특정 데이터의 빈도를 조회할 때는 각 해시 함수가 가리키는 카운터 중 가장 작은 값을 추정 빈도로 사용한다.

메모리 사용량이 적고 대용량 데이터의 빈도 분석에 유용하지만, 해시 충돌로 인해 실제 빈도보다 크게 추정할 수 있다는 단점이 있다.

### 5-1. Count-Min Sketch 클래스 구현

Count-Min Sketch의 너비(width)와 깊이(depth)를 설정하고, 2차원 카운터 배열을 생성한다.

각 행마다 서로 다른 해시 함수를 적용하여 데이터의 등장 횟수를 기록한다.

In [ ]:
import hashlib
import sys
import time
from array import array
from collections import Counter

class CountMinSketch:
    def __init__(self, width=2000, depth=5):
        self.width = width
        self.depth = depth

        # 메모리 효율적인 2차원 카운터 배열
        self.table = [
            array('Q', [0]) * width
            for _ in range(depth)
        ]

    # 행마다 다른 해시 위치 계산
    def hash_position(self, item, row):
        data = f"{row}:{item}".encode("utf-8")
        digest = hashlib.blake2b(
            data, digest_size=8
        ).digest()

        return int.from_bytes(digest, "big") % self.width

    # 데이터 빈도 증가
    def add(self, item):
        for row in range(self.depth):
            position = self.hash_position(item, row)
            self.table[row][position] += 1

    # 데이터 빈도 추정
    def estimate(self, item):
        counts = []

        for row in range(self.depth):
            position = self.hash_position(item, row)
            counts.append(self.table[row][position])

        return min(counts)

    # 카운터 배열 메모리 측정
    def memory_usage(self):
        return (
            sys.getsizeof(self.table)
            + sum(sys.getsizeof(row) for row in self.table)
        )


print("Count-Min Sketch 클래스 구현 완료!")

Count-Min Sketch 클래스 구현 완료!


### 5-2. Count-Min Sketch 실행 및 정확도 측정

전체 데이터 파일을 한 줄씩 읽어 영화 ID의 정확한 등장 횟수를 계산한다.

이후 Count-Min Sketch에도 동일한 데이터 스트림을 입력하고, 두 결과를 비교한다.

정확한 빈도와 추정 빈도의 차이를 이용하여 평균 절대 오차(MAE)와 평균 상대 오차를 계산한다.

Ground Truth는 비교를 위한 기준값이며, Count-Min Sketch 자체의 메모리 사용량과는 별도로 관리한다.

In [ ]:
# Ground Truth: 영화 ID별 정확한 등장 횟수
exact_counts = Counter()

with open(DATA_PATH, "r", encoding="latin-1") as file:
    for line in file:
        parts = line.strip().split("::")

        if len(parts) == 4:
            movie_id = int(parts[1])
            exact_counts[movie_id] += 1

print("Ground Truth 계산 완료!")
print("전체 레코드 수:", f"{sum(exact_counts.values()):,}개")
print("고유 영화 ID 수:", f"{len(exact_counts):,}개")
print("\n빈도 상위 5개 영화 ID:")

for movie_id, count in exact_counts.most_common(5):
    print(f"영화 ID {movie_id}: {count:,}회")

Ground Truth 계산 완료!
전체 레코드 수: 1,000,209개
고유 영화 ID 수: 3,706개

빈도 상위 5개 영화 ID:
영화 ID 2858: 3,428회
영화 ID 260: 2,991회
영화 ID 1196: 2,990회
영화 ID 1210: 2,883회
영화 ID 480: 2,672회


### 5-3. Count-Min Sketch 성능 측정 결과

Width를 2,000, Depth를 5로 설정한 Count-Min Sketch를 실행하고 전체 데이터 스트림을 처리한다.

실제 빈도와 추정 빈도를 비교하여 평균 절대 오차(MAE)와 평균 상대 오차를 계산한다.

또한 카운터 배열의 메모리 사용량과 전체 데이터 처리 시간 및 처리 속도를 측정하여 Count-Min Sketch의 성능을 평가한다.

In [ ]:
# Count-Min Sketch 생성
cms = CountMinSketch(width=2000, depth=5)

# 스트리밍 방식으로 데이터 입력
start_time = time.perf_counter()

processed_count = 0

with open(DATA_PATH, "r", encoding="latin-1") as file:
    for line in file:
        parts = line.strip().split("::")

        if len(parts) == 4:
            movie_id = int(parts[1])
            cms.add(movie_id)
            processed_count += 1

processing_time = time.perf_counter() - start_time

# 정확도 평가
absolute_errors = []
relative_errors = []

for movie_id, true_count in exact_counts.items():
    estimated_count = cms.estimate(movie_id)

    absolute_error = abs(estimated_count - true_count)
    relative_error = absolute_error / true_count

    absolute_errors.append(absolute_error)
    relative_errors.append(relative_error)

mae = sum(absolute_errors) / len(absolute_errors)
mean_relative_error = sum(relative_errors) / len(relative_errors) * 100

# 메모리 사용량
memory_bytes = cms.memory_usage()

print("===== Count-Min Sketch 실험 결과 =====")
print("처리 레코드 수:", f"{processed_count:,}개")
print("Width:", cms.width)
print("Depth:", cms.depth)
print("고유 영화 ID 수:", f"{len(exact_counts):,}개")
print("평균 절대 오차(MAE):", f"{mae:.4f}")
print("평균 상대 오차:", f"{mean_relative_error:.4f}%")
print("메모리 사용량:", f"{memory_bytes:,} bytes")
print("처리 시간:", f"{processing_time:.4f}초")
print("처리 속도:", f"{processed_count / processing_time:,.0f} records/sec")

===== Count-Min Sketch 실험 결과 =====
처리 레코드 수: 1,000,209개
Width: 2000
Depth: 5
고유 영화 ID 수: 3,706개
평균 절대 오차(MAE): 35.7798
평균 상대 오차: 232.8360%
메모리 사용량: 80,520 bytes
처리 시간: 9.7372초
처리 속도: 102,721 records/sec


### 5-4. Count-Min Sketch 실험 결과 분석

Count-Min Sketch를 이용하여 MovieLens 1M 데이터셋의 1,000,209개 레코드를 처리하고 영화 ID별 등장 빈도를 추정하였다.

Ground Truth와 비교한 결과 평균 절대 오차(MAE)는 35.7798, 평균 상대 오차는 232.8360%로 나타났다. 이는 해시 충돌로 인해 일부 영화 ID의 빈도가 실제보다 크게 추정되었기 때문이다. 특히 실제 등장 횟수가 적은 영화는 상대 오차가 크게 나타날 수 있다.

메모리 사용량은 80,520 bytes로 약 78.6KB였으며, 전체 데이터 처리 시간은 9.7372초, 처리 속도는 초당 약 102,721개 레코드였다.

Count-Min Sketch는 모든 데이터를 개별적으로 저장하지 않고도 데이터의 등장 횟수를 근사적으로 계산할 수 있다는 장점이 있다. 그러나 Width가 충분하지 않으면 해시 충돌이 증가하여 추정 오차가 커질 수 있다는 한계가 있다.

또한 평균 상대 오차는 영화 ID별 상대 오차를 단순 평균한 값이므로, 실제 등장 횟수가 적은 영화의 큰 상대 오차가 전체 결과에 영향을 줄 수 있다.

이번 실험에서 메모리 사용량은 Count-Min Sketch의 카운터 배열과 이를 포함하는 리스트의 크기를 기준으로 측정하였다. 따라서 Python 프로그램 전체의 메모리 사용량이 아닌, 알고리즘의 주요 자료구조가 차지하는 메모리 크기를 의미한다.

## 6. 파라미터 변경에 따른 성능 비교

Count-Min Sketch는 Width와 Depth 설정에 따라 정확도와 메모리 사용량이 달라진다.

본 실험에서는 Depth를 5로 고정하고 Width를 2,000, 5,000, 10,000으로 변경하여 성능을 비교한다.

Width가 증가하면 해시 충돌이 줄어들어 빈도 추정 오차가 감소할 수 있지만, 카운터 배열의 메모리 사용량은 증가한다.

각 설정에 대해 평균 절대 오차(MAE), 평균 상대 오차, 메모리 사용량, 처리 시간 및 처리 속도를 측정하여 파라미터 변화에 따른 Trade-off를 분석한다.

### 6-1. 파라미터별 실험 결과

Width를 2,000, 5,000, 10,000으로 변경하고 Depth는 5로 고정하여 Count-Min Sketch의 성능을 비교하였다.

각 파라미터 설정에 따른 평균 절대 오차(MAE), 평균 상대 오차, 메모리 사용량, 처리 시간 및 처리 속도를 측정하였다.

모든 실험은 동일한 MovieLens 1M 데이터셋을 사용하였으며, Width 외의 조건은 동일하게 유지하여 Width 변화에 따른 성능 차이를 비교하였다.

In [ ]:
# Count-Min Sketch 파라미터별 성능 실험

def run_cms_experiment(width, depth=5):
    cms_test = CountMinSketch(width=width, depth=depth)

    start_time = time.perf_counter()

    # 스트리밍 방식으로 데이터 입력
    with open(DATA_PATH, "r", encoding="latin-1") as file:
        for line in file:
            parts = line.strip().split("::")

            if len(parts) == 4:
                movie_id = int(parts[1])
                cms_test.add(movie_id)

    processing_time = time.perf_counter() - start_time

    # Ground Truth와 비교
    absolute_errors = []
    relative_errors = []

    for movie_id, true_count in exact_counts.items():
        estimated_count = cms_test.estimate(movie_id)

        error = abs(estimated_count - true_count)

        absolute_errors.append(error)
        relative_errors.append(error / true_count)

    mae = sum(absolute_errors) / len(absolute_errors)
    mre = sum(relative_errors) / len(relative_errors) * 100

    return {
        "Width": width,
        "Depth": depth,
        "MAE": mae,
        "평균 상대 오차(%)": mre,
        "메모리(bytes)": cms_test.memory_usage(),
        "처리 시간(초)": processing_time,
        "처리 속도(records/sec)": sum(exact_counts.values()) / processing_time
}


# Width별 실험
experiment_results = []

for width in [2000, 5000, 10000]:
    result = run_cms_experiment(width)
    experiment_results.append(result)

# 실험 결과 표 출력
import pandas as pd

experiment_df = pd.DataFrame(experiment_results)

print("===== Count-Min Sketch 파라미터 비교 결과 =====")

display(
    experiment_df.style
    .hide(axis="index")
    .format({
        "MAE": "{:.4f}",
        "평균 상대 오차(%)": "{:.4f}",
        "메모리(bytes)": "{:,.0f}",
        "처리 시간(초)": "{:.4f}",
        "처리 속도(records/sec)": "{:,.0f}"
    })
)

===== Count-Min Sketch 파라미터 비교 결과 =====


Width,Depth,MAE,평균 상대 오차(%),메모리(bytes),처리 시간(초),처리 속도(records/sec)
2000,5,35.7798,232.8360,"80,520",8.8636,"112,845"
5000,5,1.9428,13.6782,"200,520",9.9974,"100,047"
10000,5,0.1198,0.0942,"400,520",9.8561,"101,481"


### 6-2. 파라미터 변경 실험 결과 분석

평균 상대 오차는 전체 영화 ID 3,706개의 상대 오차를 동일한 비중으로 평균한 값이다.

실험 결과 Width가 증가할수록 평균 절대 오차와 평균 상대 오차가 감소하는 것을 확인하였다.

Width가 2,000일 때 평균 절대 오차는 35.7798, 평균 상대 오차는 232.8360%였다. Width를 5,000으로 증가시키자 평균 절대 오차는 1.9428, 평균 상대 오차는 13.6782%로 감소하였다. Width가 10,000일 때는 평균 절대 오차가 0.1198, 평균 상대 오차가 0.0942%까지 감소하여 가장 높은 추정 정확도를 보였다.

반면 메모리 사용량은 Width가 2,000일 때 80,520 bytes에서 Width가 5,000일 때 200,520 bytes, Width가 10,000일 때 400,520 bytes로 증가하였다.

이를 통해 Count-Min Sketch는 Width를 크게 설정할수록 해시 충돌이 줄어들어 빈도 추정 정확도가 향상되지만, 더 많은 메모리가 필요하다는 Trade-off를 확인하였다.

처리 시간은 각각 8.8636초, 9.9974초, 9.8561초로 나타났으며, 처리 속도는 각각 초당 약 112,845개, 100,047개, 101,481개 레코드로 측정되었다. Width 증가에 따라 처리 시간이 일정하게 증가하거나 감소하는 패턴은 나타나지 않았다. 이는 실제 처리 시간이 알고리즘의 파라미터뿐 아니라 실행 환경과 시스템 상태의 영향을 받기 때문이다. 따라서 Width 증가가 항상 처리 속도를 향상시킨다고 단정할 수는 없다.

따라서 Count-Min Sketch의 Width는 무조건 크게 설정하기보다는 요구되는 정확도와 사용 가능한 메모리 사이의 균형을 고려하여 결정해야 한다.

## 7. 알고리즘 종합 비교 및 결론

### 7-1. Bloom Filter와 Count-Min Sketch 성능 비교

Bloom Filter와 Count-Min Sketch는 모두 대용량 데이터 스트림을 효율적으로 처리하기 위한 확률적 자료구조이지만, 목적과 활용 방식에 차이가 있다.

Bloom Filter는 데이터의 존재 여부를 빠르게 확인하는 데 적합하며, Count-Min Sketch는 데이터의 등장 횟수를 근사적으로 추정하는 데 적합하다.

두 알고리즘의 실험 결과를 비교하면 다음과 같다.

| 비교 항목 | Bloom Filter | Count-Min Sketch |
|---|---:|---:|
| 주요 목적 | 데이터 존재 여부 확인 | 데이터 빈도 추정 |
| 메모리 사용량 | 4,498 bytes | 80,520 bytes |
| 처리 시간 | 5.5140초 | 9.7372초 |
| 처리 속도 | 181,396 records/sec | 102,721 records/sec |
| 정확도 지표 1 | FPR 1.1592% | MAE 35.7798 |
| 정확도 지표 2 | - | 평균 상대 오차 232.8360% |

※ Bloom Filter는 데이터 존재 여부 판정의 오탐률(FPR)을 측정하였으며, Count-Min Sketch는 실제 빈도와 추정 빈도 사이의 평균 절대 오차(MAE)와 평균 상대 오차를 측정하였다. 두 알고리즘은 목적과 정확도 측정 기준이 다르므로 정확도 수치를 직접 비교하기보다는 각 알고리즘의 목적에 맞는 지표로 평가하였다.

실험 결과, Bloom Filter는 Count-Min Sketch보다 적은 메모리를 사용하고 더 빠른 처리 속도를 보였다. 이는 Bloom Filter가 데이터의 존재 여부만 확인하는 자료구조인 반면, Count-Min Sketch는 데이터의 등장 횟수를 기록하고 추정해야 하기 때문이다.

다만 두 알고리즘은 목적이 서로 다르므로 단순히 성능 수치만으로 어느 알고리즘이 더 우수하다고 판단하기는 어렵다. 따라서 실제 활용 환경에서 필요한 기능과 정확도 수준에 따라 적절한 알고리즘을 선택해야 한다.

### 7-2. 정확도와 메모리 사용량의 Trade-off 분석

Bloom Filter는 비트 배열의 크기와 해시 함수 개수를 조정하여 False Positive 발생 가능성을 낮출 수 있다. 하지만 더 낮은 오탐률을 얻기 위해서는 일반적으로 더 많은 메모리가 필요하다.

Count-Min Sketch 역시 Width와 Depth를 조정하여 해시 충돌 가능성을 줄이고 빈도 추정 정확도를 향상시킬 수 있다. 이번 실험에서는 Depth를 5로 고정하고 Width를 2,000에서 10,000으로 증가시켰다. 그 결과 평균 절대 오차는 35.7798에서 0.1198로 감소하였으며, 평균 상대 오차도 232.8360%에서 0.0942%로 감소하였다.

반면 메모리 사용량은 Width가 2,000일 때 80,520 bytes에서 Width가 10,000일 때 400,520 bytes로 증가하였다.

Bloom Filter의 파라미터 변경 실험에서도 목표 FPR을 5.0%에서 0.1%로 낮추자 메모리 사용량이 2,946 bytes에서 6,718 bytes로 증가하였다. 동시에 실제 FPR은 4.8686%에서 0.3091%로 감소하였다. 이를 통해 Bloom Filter 역시 더 낮은 오탐률을 얻기 위해 더 많은 메모리가 필요하다는 점을 확인하였다.

**표 7-2. Count-Min Sketch의 Width 변경에 따른 정확도 및 메모리 사용량 비교**

| Width | MAE | 평균 상대 오차 | 메모리 사용량 |
|---:|---:|---:|---:|
| 2,000 | 35.7798 | 232.8360% | 80,520 bytes |
| 5,000 | 1.9428 | 13.6782% | 200,520 bytes |
| 10,000 | 0.1198 | 0.0942% | 400,520 bytes |

이를 통해 두 알고리즘 모두 메모리 사용량과 정확도 사이에 Trade-off가 존재한다는 점을 확인하였다. 따라서 실제 서비스에서는 필요한 정확도와 사용 가능한 메모리를 고려하여 적절한 파라미터를 설정해야 한다.

### 7-3. 실제 서비스 환경에서의 활용 가능성

Bloom Filter는 회원 가입 시 이미 사용 중인 아이디인지 빠르게 확인하거나, 웹 크롤링에서 이미 방문한 URL인지 검사하는 상황에 활용할 수 있다. 적은 메모리로 빠르게 데이터의 존재 여부를 확인할 수 있기 때문이다. 다만 False Positive가 발생할 수 있으므로 최종적인 존재 여부 확인이 필요한 경우에는 별도의 검증 과정이 필요하다.

Count-Min Sketch는 서비스 로그에서 특정 이벤트나 상품의 등장 횟수를 추정하거나, 실시간으로 많이 발생하는 검색어와 요청 패턴을 파악하는 상황에 적합하다. 모든 데이터를 개별적으로 저장하지 않고도 빈도 정보를 근사적으로 관리할 수 있다는 장점이 있다.

따라서 데이터의 존재 여부를 확인하는 것이 목적이라면 Bloom Filter를, 데이터의 빈도와 발생량을 분석하는 것이 목적이라면 Count-Min Sketch를 선택하는 것이 적절하다. 특히 대규모 서비스 로그에서 이벤트 발생 횟수나 인기 검색어를 실시간으로 분석하려는 경우에는 Count-Min Sketch가 유용하게 활용될 수 있다.

### 7-4. 최종 결론

이번 실험에서는 MovieLens 1M 데이터셋의 1,000,209개 레코드를 활용하여 Bloom Filter와 Count-Min Sketch를 직접 구현하고 성능을 비교하였다.

실험을 통해 Bloom Filter는 적은 메모리와 빠른 처리 속도를 바탕으로 데이터의 존재 여부를 효율적으로 확인할 수 있다는 점을 확인하였다. 반면 Count-Min Sketch는 일부 추정 오차가 발생하지만, 데이터의 등장 횟수를 근사적으로 계산할 수 있다는 장점이 있었다.

또한 Bloom Filter의 목표 FPR을 낮추는 실험에서는 실제 오탐률이 감소하는 대신 메모리 사용량이 증가하는 것을 확인하였다. Count-Min Sketch의 Width를 변경하는 실험에서도 Width가 증가할수록 해시 충돌이 감소하고 빈도 추정 정확도가 향상되지만, 메모리 사용량도 함께 증가한다는 사실을 확인하였다.

결론적으로 스트리밍 알고리즘은 모든 상황에서 정확도와 메모리 효율을 동시에 극대화하기 어렵기 때문에, 데이터 처리 목적과 서비스 환경에 맞는 알고리즘 및 파라미터를 선택하는 것이 중요하다.

### 7-5. 최종 분석 질문에 대한 답변

**1. 정확도와 메모리 사이에는 어떤 Trade-off가 있었는가?**

Count-Min Sketch 실험에서 Width를 2,000에서 10,000으로 증가시키자 평균 절대 오차는 35.7798에서 0.1198로 감소하였다. 반면 메모리 사용량은 80,520 bytes에서 400,520 bytes로 증가하였다.

Bloom Filter 역시 목표 FPR을 5.0%에서 0.1%로 낮추자 실제 FPR은 4.8686%에서 0.3091%로 감소하였으며, 메모리 사용량은 2,946 bytes에서 6,718 bytes로 증가하였다.

이를 통해 두 알고리즘 모두 더 높은 정확도를 얻기 위해서는 일반적으로 더 많은 메모리가 필요하다는 Trade-off를 확인하였다.

**2. 파라미터 증가가 항상 성능 향상으로 이어졌는가?**

파라미터 증가가 모든 성능 지표를 향상시키지는 않았다. Count-Min Sketch의 Width를 증가시키자 추정 오차는 크게 감소했지만 메모리 사용량은 증가하였다. 또한 처리 시간은 각각 8.8636초, 9.9974초, 9.8561초로 나타났다. Width 증가에 따라 처리 시간이 일정하게 감소하거나 증가하는 패턴은 나타나지 않았다. 이는 실행 환경과 시스템 상태의 영향을 받을 수 있기 때문이다.

따라서 파라미터 증가는 정확도 향상에 도움을 줄 수 있지만, 전체적인 성능 향상을 항상 보장하지는 않는다.

**3. 어떤 알고리즘이 가장 실용적이라고 판단되는가?**

두 알고리즘의 실용성은 데이터 처리 목적에 따라 달라진다. 데이터의 존재 여부를 빠르게 확인하는 상황에서는 Bloom Filter가 가장 실용적이라고 판단된다. 본 실험에서 Bloom Filter는 약 4.4KB의 적은 메모리로 1,000,209개 레코드를 처리했으며, 초당 약 181,396개의 레코드를 처리하였다.

반면 데이터의 등장 횟수를 분석해야 하는 상황에서는 Count-Min Sketch가 더 적합하다. 따라서 알고리즘의 실용성은 단순한 처리 속도나 메모리 사용량보다는 실제 활용 목적을 기준으로 판단해야 한다.

**4. 실제 서비스 로그 분석에 적용한다면 어떤 알고리즘을 선택할 것인가?**

실제 서비스 로그에서 특정 이벤트의 발생 횟수나 인기 검색어를 실시간으로 분석하려는 목적이라면 Count-Min Sketch를 선택할 것이다. 모든 로그를 개별적으로 저장하지 않고도 빈도를 근사적으로 추정할 수 있기 때문이다. 반면 중복 요청이나 이미 처리한 이벤트인지 확인하는 것이 목적이라면 Bloom Filter를 선택하는 것이 적절하다.

따라서 실제 서비스 로그 분석에서는 분석 목적에 따라 두 알고리즘을 단독으로 사용하거나 함께 활용하는 것도 가능하다.